In [1]:
import sys
sys.path.append('../')
sys.path.append('../../')
import torch
from MINAR.ComputationGraph import ComputationGraph, Circuit
from model.CustomLosses import MultiplicativeLoss
import torch_geometric as pyg
import networkx as nx

import numpy as np
from model.MinAggGNN import MinAggGNN
import matplotlib.pyplot as plt


seeds = [0, 1, 2, 3, 4]
device = torch.device('cuda')
L = 2

test_data = torch.load('../data/test_data.pt', map_location=device, weights_only=False)
test_loader = pyg.loader.DataLoader(test_data, batch_size = len(test_data))
criterion = MultiplicativeLoss()
mse_criterion = torch.nn.MSELoss()
num_reachable_test_nodes = sum([data.reachable.sum() for data in test_loader])

corrupted_data = torch.load('../data/test_data.pt', map_location=device, weights_only=False)
for data_corr in corrupted_data:
    data_corr.x = torch.zeros_like(data_corr.x, device=device)
    data_corr.x[0] = 0.
    data_corr.edge_attr = torch.zeros_like(data_corr.edge_attr, device=device)

def load_circuit(model):
    G = ComputationGraph(model)
    G.add_inputs({'edge_attr' : [1, model.convs[0].agg_mlp.lins[0].weight[:,-1]],
                'input_self' : [3, model.convs[0].up_mlp.lins[0].weight[:,-1]]})
    G.add_residual_connections({'edge_attr' : [5, model.convs[1].agg_mlp.lins[0].weight[:,-1].reshape(1,-1).cpu().detach()]})
    G.add_residual_connections({4 : [7, model.convs[1].up_mlp.lins[0].weight[:,-8:].T.cpu().detach()]})
    G.calculate_scores(test_data, corrupted_data, mse_criterion, which = 'EAP-IG', steps=20)
    C = Circuit(model, G, K=10, key='EAP-IG')
    return C

c:\Users\heje197\AppData\Local\miniconda3\envs\minar\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_losses = []
circuit_losses = []
circuit_ablated_losses = []

model = MinAggGNN(1, 8, L, 1, edge_dim = 1)
for seed in seeds:
    state_dict = torch.load(f'../model_progress/bellman_ford/seed_{seed}/model_final.pt')
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)
    circuit = load_circuit(model)

    model_loss = 0
    circuit_loss = 0
    circuit_ablated_loss = 0
    for data in test_loader:
        data = data.to(device)

        model_out = model(data.x, data.edge_index, edge_attr=data.edge_attr)
        circuit_out = circuit.forward(data)
        circuit_ablated_out = circuit.ablate_circuit(data)

        model_loss += criterion(model_out[data.reachable].flatten(), data.y[data.reachable]).detach().item() / num_reachable_test_nodes
        circuit_loss += criterion(circuit_out[data.reachable].flatten(), data.y[data.reachable]).detach().item() / num_reachable_test_nodes
        circuit_ablated_loss += criterion(circuit_ablated_out[data.reachable].flatten(), data.y[data.reachable]).detach().item() / num_reachable_test_nodes

    model_losses.append(model_loss)
    circuit_losses.append(circuit_loss)
    circuit_ablated_losses.append(circuit_ablated_loss)

In [3]:
print(f"{'seed':>5} {'model_loss':>10} {'circuit_loss':>12} {'ablated_loss':>12}")
for seed, model_loss, circuit_loss, ablated_loss in zip(seeds, model_losses, circuit_losses, circuit_ablated_losses):
    print(f"{seed:>5} {model_loss:>10.4f} {circuit_loss:>12.4f} {ablated_loss:>12.4f}")

print(f"\n{'metric':>12} {'mean':>12} {'std':>12}")
print(f"{'model_loss':>12} {torch.tensor(model_losses).mean().item():>12.4f} {torch.tensor(model_losses).std().item():>12.4f}")
print(f"{'circuit_loss':>12} {torch.tensor(circuit_losses).mean().item():>12.4f} {torch.tensor(circuit_losses).std().item():>12.4f}")
print(f"{'ablated_loss':>12} {torch.tensor(circuit_ablated_losses).mean().item():>12.4f} {torch.tensor(circuit_ablated_losses).std().item():>12.4f}")

 seed model_loss circuit_loss ablated_loss
    0     0.0578       0.0545   21870.6836
    1     0.0577       0.0543  698918.0000
    2     0.0579       0.0540   35846.6523
    3     0.0569       0.0543   71579.8516
    4     0.0589       0.0542   21196.0332

      metric         mean          std
  model_loss       0.0578       0.0007
circuit_loss       0.0543       0.0002
ablated_loss  169882.2500  296446.7812
